In [ ]:
!pip install nltk scikit-learn

In [ ]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
print(df.columns)

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')


In [ ]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score, classification_report

# ---------------------------------------------------
# DOWNLOAD NLTK DATA
# ---------------------------------------------------

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# ---------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------

df = pd.read_csv("support_tickets.csv")

# ---------------------------------------------------
# COMBINE SUBJECT + DESCRIPTION
# ---------------------------------------------------

df['combined_text'] = (
    df['Ticket Subject'].astype(str)
    + " "
    + df['Ticket Description'].astype(str)
)

# ---------------------------------------------------
# TEXT CLEANING
# ---------------------------------------------------

stop_words = set(stopwords.words('english'))

def clean_text(text):

    text = text.lower()

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

# Apply cleaning
df['clean_text'] = df['combined_text'].apply(clean_text)

# ---------------------------------------------------
# TF-IDF VECTORIZATION
# ---------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X = tfidf.fit_transform(df['clean_text'])

# ===================================================
# CATEGORY CLASSIFICATION
# ===================================================

y_category = df['Ticket Type']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_category,
    test_size=0.2,
    random_state=42
)

# Better model for text classification
model_category = MultinomialNB()

model_category.fit(X_train, y_train)

# Predictions
y_pred = model_category.predict(X_test)

print("\nCATEGORY CLASSIFICATION RESULTS")

print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

# ===================================================
# PRIORITY PREDICTION
# ===================================================

y_priority = df['Ticket Priority']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X,
    y_priority,
    test_size=0.2,
    random_state=42
)

model_priority = MultinomialNB()

model_priority.fit(X_train_p, y_train_p)

# Predictions
y_pred_p = model_priority.predict(X_test_p)

print("\nPRIORITY PREDICTION RESULTS")

print("Accuracy:", accuracy_score(y_test_p, y_pred_p))

print(classification_report(y_test_p, y_pred_p))

# ===================================================
# TEST NEW TICKET
# ===================================================

new_ticket = [
    "Payment deducted but order not confirmed and refund not received"
]

# Clean text
cleaned = clean_text(new_ticket[0])

# Convert into TF-IDF vector
vector = tfidf.transform([cleaned])

# Predictions
predicted_category = model_category.predict(vector)

predicted_priority = model_priority.predict(vector)

print("\nNEW TICKET PREDICTION")

print("Ticket:", new_ticket[0])

print("Predicted Category:", predicted_category[0])

print("Predicted Priority:", predicted_priority[0])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



CATEGORY CLASSIFICATION RESULTS
Accuracy: 0.19008264462809918
                      precision    recall  f1-score   support

     Billing inquiry       0.18      0.10      0.13       357
Cancellation request       0.18      0.18      0.18       327
     Product inquiry       0.17      0.18      0.18       316
      Refund request       0.19      0.23      0.20       345
     Technical issue       0.22      0.27      0.24       349

            accuracy                           0.19      1694
           macro avg       0.19      0.19      0.19      1694
        weighted avg       0.19      0.19      0.19      1694


PRIORITY PREDICTION RESULTS
Accuracy: 0.25206611570247933
              precision    recall  f1-score   support

    Critical       0.24      0.29      0.27       411
        High       0.24      0.23      0.24       409
         Low       0.25      0.23      0.24       415
      Medium       0.28      0.26      0.27       459

    accuracy                           0.25  